# Gemma-3-270M-IT | Medical QA | QLoRA Finetuning
**Kaggle T4 GPU** | Combined dataset | Pre/Post evaluation with BLEU, ROUGE, F1

## Cell 1 — Install

In [1]:
%%capture
!pip install -q "transformers>=4.50.0"
!pip install -q "tokenizers>=0.21.0"
!pip install -q "huggingface_hub>=0.23.0"
!pip install -q "peft>=0.11.0"
!pip install -q "trl>=0.9.0"
!pip install -q "bitsandbytes>=0.43.0"
!pip install -q "accelerate>=0.30.0"
!pip install -q "datasets>=2.19.0"
!pip install -q sacrebleu==2.4.3
!pip install -q rouge-score==0.1.2
!pip install -q nltk
!pip install -q sentencepiece
print("Done")

In [2]:
import socket

def check_internet(host="8.8.8.8", port=53, timeout=3):
    """
    Check if the internet is accessible by connecting to Google's Public DNS.
    Returns True if reachable, False otherwise.
    """
    try:
        # Create a socket connection to the host/port
        socket.setdefaulttimeout(timeout)
        socket.socket(socket.AF_INET, socket.SOCK_STREAM).connect((host, port))
        return True
    except socket.error:
        return False

# Execution
if check_internet():
    print("Internet is accessible.")
else:
    print("Internet is NOT accessible.")

Internet is accessible.


## Cell 2 — Imports

In [ ]:
!nvidia-smi

In [4]:
import torch
# This forces CUDA initialization without seeding
x = torch.tensor([1.0]).cuda() 
print("CUDA Init Successful")


CUDA Init Successful


In [5]:
import torch
print(f"Device count: {torch.cuda.device_count()}")

for i in range(torch.cuda.device_count()):
    try:
        device = f"cuda:{i}"
        x = torch.ones(1).to(device)
        print(f"GPU {i} ({torch.cuda.get_device_name(i)}): OK")
    except Exception as e:
        print(f"GPU {i} ({torch.cuda.get_device_name(i)}): FAILED - {e}")

Device count: 2
GPU 0 (Tesla T4): OK
GPU 1 (Tesla T4): OK


In [6]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [7]:
import os, gc, csv, json, time, random, logging, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple

warnings.filterwarnings("ignore")

import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainerCallback, TrainerState, TrainerControl,
    set_seed,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel, TaskType
from trl import SFTTrainer, SFTConfig

import sacrebleu
from rouge_score import rouge_scorer as rouge_scorer_lib
import nltk
from nltk.tokenize import word_tokenize
nltk.download("punkt",     quiet=True)
nltk.download("punkt_tab", quiet=True)

# ── Reproducibility ──────────────────────────────────────────────
SEED = 42
# Replace set_seed(SEED) with this to test:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED) # Only seeds the default GPU (0)

# ── Logger ───────────────────────────────────────────────────────
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s [%(levelname)s] %(message)s",
                    datefmt="%H:%M:%S")
log = logging.getLogger("medical_ft")

log.info(f"torch        : {torch.__version__}")
log.info(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    log.info(f"GPU          : {torch.cuda.get_device_name(0)}")
    log.info(f"VRAM         : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

import transformers, peft, trl
log.info(f"transformers : {transformers.__version__}")
log.info(f"peft         : {peft.__version__}")
log.info(f"trl          : {trl.__version__}")
print("Imports OK")

19:22:40 [INFO] torch        : 2.10.0+cu128
19:22:40 [INFO] CUDA         : True
19:22:40 [INFO] GPU          : Tesla T4
19:22:40 [INFO] VRAM         : 15.6 GB
19:22:40 [INFO] transformers : 5.0.0
19:22:40 [INFO] peft         : 0.18.1
19:22:40 [INFO] trl          : 1.3.0


Imports OK


## Cell 3 — Config

In [8]:
# ── Paths ────────────────────────────────────────────────────────
WORK        = Path("/kaggle/working")
LOG_DIR     = WORK / "logs";          LOG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = WORK / "results";       RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR    = WORK / "checkpoints";   CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── Model ────────────────────────────────────────────────────────
MODEL_ID      = "google/gemma-3-270m-it"
MAX_SEQ_LEN   = 512

# ── Dataset samples ──────────────────────────────────────────────
N_PUBMEDQA    = 1000   # full labeled split
N_USMLE       = 4000
N_ALPACA      = 8000
N_COMPMEDQA   = 3000
# Comprehensive Medical Q&A — attach to Kaggle notebook as input dataset
COMPMEDQA_CSV = "/kaggle/input/datasets/thedevastator/comprehensive-medical-q-a-dataset/train.csv"

# ── QLoRA ────────────────────────────────────────────────────────
BNB_CFG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

LORA_CFG = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# ── Training ─────────────────────────────────────────────────────
TRAIN_CFG = dict(
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,   # effective batch = 16
    learning_rate=2e-4,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    logging_steps=50,
    save_strategy="epoch",
    save_total_limit=1,
    fp16=True,
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
    weight_decay=0.001,
    report_to="none",
    dataloader_num_workers=2,
)

# ── Evaluation ───────────────────────────────────────────────────
N_EVAL_SAMPLES = 200   # samples per evaluation pass
MAX_NEW_TOKENS = 128

# ── Metrics objects (no HF hub download needed) ──────────────────
ROUGE = rouge_scorer_lib.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

log.info("Config ready.")

19:22:40 [INFO] Using default tokenizer.
19:22:40 [INFO] Config ready.


## Cell 4 — HuggingFace Login
> **Required**: accept the Gemma licence at https://huggingface.co/google/gemma-3-270m-it  
> then add your HF token in **Kaggle → Add-ons → Secrets → HF_TOKEN**

In [9]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets = UserSecretsClient()
login(token=secrets.get_secret("HF_TOKEN"), add_to_git_credential=False)
log.info("HuggingFace login OK")

19:22:40 [INFO] HTTP Request: GET https://huggingface.co/api/whoami-v2 "HTTP/1.1 200 OK"
19:22:40 [INFO] HuggingFace login OK


## Cell 5 — Dataset Loading & Unified Format
Every sample → `<instruction>...</instruction><context>...</context><input>...</input><output>...</output>`

In [10]:
def fmt(instruction: str, context: str, inp: str, out: str) -> str:
    """Convert any QA pair to the standard 4-tag generation format."""
    return (
        f"<instruction>{instruction.strip()}</instruction>\n"
        f"<context>{context.strip()}</context>\n"
        f"<input>{inp.strip()}</input>\n"
        f"<output>{out.strip()}</output>"
    )


# ────────────────────────────────────────────────────────────────
def load_pubmedqa() -> List[Dict]:
    log.info("Loading PubMedQA …")
    ds = load_dataset("qiaojin/PubMedQA", "pqa_labeled",
                      split="train", trust_remote_code=True)
    ds = ds.shuffle(seed=SEED).select(range(min(N_PUBMEDQA, len(ds))))
    out = []
    for r in ds:
        ctx   = " ".join((r["context"].get("contexts") or [])[:][:3]) or "N/A"
        ans   = f"{r['long_answer']} Decision: {r['final_decision']}.".strip()
        out.append({"text": fmt(
            "Answer the biomedical question using the PubMed context. "
            "Give a detailed answer then state yes/no/maybe.",
            ctx, r["question"], ans), "answer": ans, "source": "pubmedqa"})
    log.info(f"  PubMedQA      : {len(out)} samples")
    return out


def load_usmle() -> List[Dict]:
    log.info("Loading MedQA-USMLE …")
    ds = load_dataset("GBaker/MedQA-USMLE-4-options",
                      split="train", trust_remote_code=True)
    ds = ds.shuffle(seed=SEED).select(range(min(N_USMLE, len(ds))))
    out = []
    for r in ds:
        opts    = r["options"]                                   # {A:…, B:…, C:…, D:…}
        ctx     = "\n".join(f"{k}. {v}" for k, v in opts.items())
        key     = r.get("answer_idx", "")
        ans     = f"{key}. {opts.get(key, r.get('answer', ''))}"
        out.append({"text": fmt(
            "Select the single best answer for this USMLE question and justify briefly.",
            ctx, r["question"], ans), "answer": ans, "source": "medqa_usmle"})
    log.info(f"  MedQA-USMLE   : {len(out)} samples")
    return out


def load_alpaca() -> List[Dict]:
    log.info("Loading AlpaCare-MedInstruct-52k …")
    ds = load_dataset("lavita/AlpaCare-MedInstruct-52k",
                      split="train", trust_remote_code=True)
    ds = ds.shuffle(seed=SEED).select(range(min(N_ALPACA, len(ds))))
    out = []
    for r in ds:
        ans = (r.get("output") or "").strip()
        if not ans:
            continue
        out.append({"text": fmt(
            r.get("instruction") or "Provide a helpful medical response.",
            "Medical instruction-following.",
            r.get("input") or "N/A", ans),
            "answer": ans, "source": "alpaca_care"})
    log.info(f"  AlpaCare      : {len(out)} samples")
    return out


def load_compmedqa() -> List[Dict]:
    p = Path(COMPMEDQA_CSV)
    if not p.exists():
        log.warning(f"Comprehensive Medical Q&A not found at {p} — skipping.")
        return []
    log.info("Loading Comprehensive Medical Q&A …")
    df = pd.read_csv(p)
    df.columns = [c.strip().lower() for c in df.columns]
    qc = next((c for c in df.columns if "question" in c), None)
    ac = next((c for c in df.columns if "answer"   in c), None)
    if not qc or not ac:
        log.warning(f"Cannot find question/answer columns. Columns: {list(df.columns)}")
        return []
    df = df[[qc, ac]].dropna().sample(frac=1, random_state=SEED).head(N_COMPMEDQA)
    out = []
    for _, row in df.iterrows():
        q, a = str(row[qc]).strip(), str(row[ac]).strip()
        out.append({"text": fmt(
            "Answer the following medical question clearly and accurately.",
            "General medical knowledge.", q, a),
            "answer": a, "source": "comprehensive_medqa"})
    log.info(f"  Comprehensive : {len(out)} samples")
    return out


print("Loader functions defined.")

Loader functions defined.


## Cell 6 — Build Combined Dataset (80/20 split)

In [ ]:
all_samples: List[Dict] = []
all_samples.extend(load_pubmedqa())
all_samples.extend(load_usmle())
all_samples.extend(load_alpaca())
all_samples.extend(load_compmedqa())

# ── Shuffle & split ──────────────────────────────────────────────
random.shuffle(all_samples)
n_train = int(len(all_samples) * 0.8)
train_samples = all_samples[:n_train]
test_samples  = all_samples[n_train:]

# ── Source breakdown in test set ─────────────────────────────────
from collections import Counter
src_counts = Counter(s["source"] for s in all_samples)

print("\n" + "="*52)
print(f"  Total samples  : {len(all_samples)}")
print(f"  Train          : {len(train_samples)}")
print(f"  Test           : {len(test_samples)}")
print("  --- per source ---")
for src, cnt in src_counts.items():
    print(f"  {src:<25}: {cnt}")
print("="*52)

# ── HuggingFace Dataset for training (needs only 'text' column) ──
train_hf = Dataset.from_dict({"text": [s["text"] for s in train_samples]})

# ── Save test set answers for evaluation ─────────────────────────
with open(RESULTS_DIR / "test_samples.json", "w") as f:
    json.dump(test_samples, f, indent=2)

log.info(f"Combined dataset ready — {len(train_hf)} training rows.")

In [ ]:
import os
from pathlib import Path

# 1. Set up Kaggle output paths
# /kaggle/working is the only writable directory that persists for download
KAGGLE_OUTPUT = Path("/kaggle/working/medical_ft_data")
KAGGLE_OUTPUT.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = KAGGLE_OUTPUT / "train_hf"
TEST_JSON_PATH = KAGGLE_OUTPUT / "test_samples.json"

# 2. Save the Hugging Face Dataset (Arrow format)
# This is ideal for Kaggle because it allows memory-mapping during training
train_hf.save_to_disk(str(TRAIN_PATH))

# 3. Save the test set (JSON)
with open(TEST_JSON_PATH, "w") as f:
    json.dump(test_samples, f, indent=2)

log.info(f"Successfully saved to Kaggle Working Directory: {KAGGLE_OUTPUT}")

# 4. Optional: Create a zip file for easy local download
# Kaggle's UI can be slow with folders; zipping makes it a one-click download
import shutil
shutil.make_archive("/kaggle/working/processed_data", 'zip', KAGGLE_OUTPUT)
log.info("Dataset zipped for download: /kaggle/working/processed_data.zip")

## Cell 7 — Metric Functions

In [13]:
def token_f1(pred: str, ref: str) -> float:
    """SQuAD-style token-overlap F1."""
    p_toks = set(word_tokenize(pred.lower()))
    r_toks = set(word_tokenize(ref.lower()))
    if not p_toks or not r_toks:
        return 0.0
    common = p_toks & r_toks
    if not common:
        return 0.0
    prec = len(common) / len(p_toks)
    rec  = len(common) / len(r_toks)
    return 2 * prec * rec / (prec + rec)


def compute_metrics(predictions: List[str], references: List[str]) -> Dict:
    """Return BLEU (0-1), ROUGE-1/2/L (F1), token-F1 averaged over the batch."""
    pairs = [(p.strip(), r.strip())
             for p, r in zip(predictions, references)
             if p.strip() and r.strip()]
    if not pairs:
        return {k: 0.0 for k in ["bleu", "rouge1", "rouge2", "rougeL", "f1"]}

    preds, refs = zip(*pairs)

    # BLEU
    try:
        bleu = sacrebleu.corpus_bleu(list(preds), [list(refs)]).score / 100.0
    except Exception:
        bleu = 0.0

    # ROUGE
    r1s, r2s, rLs = [], [], []
    for p, r in zip(preds, refs):
        sc = ROUGE.score(r, p)
        r1s.append(sc["rouge1"].fmeasure)
        r2s.append(sc["rouge2"].fmeasure)
        rLs.append(sc["rougeL"].fmeasure)

    # Token F1
    f1s = [token_f1(p, r) for p, r in zip(preds, refs)]

    return {
        "bleu":   round(bleu,              4),
        "rouge1": round(np.mean(r1s),      4),
        "rouge2": round(np.mean(r2s),      4),
        "rougeL": round(np.mean(rLs),      4),
        "f1":     round(np.mean(f1s),      4),
        "n":      len(pairs),
    }


def extract_prompt(text: str) -> str:
    """Strip <output> content → prompt that ends at the opening tag."""
    return text[:text.index("<output>") + len("<output>")] if "<output>" in text else text


def extract_generated(text: str) -> str:
    """Pull text after <output> (before </output> if present)."""
    if "<output>" in text:
        after = text[text.index("<output>") + len("<output>"):]
        return after[:after.index("</output>")].strip() if "</output>" in after else after.strip()
    return text.strip()


print("Metric functions defined.")

Metric functions defined.


## Cell 8 — Model Loader & CSV Training Logger

In [14]:
# ── CSV logger ───────────────────────────────────────────────────
TRAIN_LOG_PATH = LOG_DIR / "training_log.csv"
EVAL_LOG_PATH  = LOG_DIR / "eval_log.csv"

def _init_csv(path: Path, fields: List[str]):
    if not path.exists():
        with open(path, "w", newline="") as f:
            csv.DictWriter(f, fieldnames=fields).writeheader()

def _append_csv(path: Path, fields: List[str], row: Dict):
    row["ts"] = datetime.now().strftime("%H:%M:%S")
    with open(path, "a", newline="") as f:
        csv.DictWriter(f, fieldnames=fields).writerow(
            {k: row.get(k, "") for k in fields})

TRAIN_FIELDS = ["ts", "step", "epoch", "loss", "lr", "elapsed_s"]
EVAL_FIELDS  = ["ts", "phase", "bleu", "rouge1", "rouge2", "rougeL", "f1", "n"]
_init_csv(TRAIN_LOG_PATH, TRAIN_FIELDS)
_init_csv(EVAL_LOG_PATH,  EVAL_FIELDS)


class StepLogger(TrainerCallback):
    def __init__(self):
        self._t0 = time.time()
    def on_log(self, args, state: TrainerState, control: TrainerControl, logs=None, **kw):
        if logs and "loss" in logs:
            _append_csv(TRAIN_LOG_PATH, TRAIN_FIELDS, {
                "step":      state.global_step,
                "epoch":     round(state.epoch or 0, 3),
                "loss":      round(logs["loss"], 5),
                "lr":        logs.get("learning_rate", ""),
                "elapsed_s": round(time.time() - self._t0, 1),
            })
            log.info(f"step {state.global_step:>5} | "
                     f"loss {logs['loss']:.4f} | "
                     f"epoch {state.epoch:.2f}")


# ── Model loader ─────────────────────────────────────────────────
def load_model_and_tokenizer(lora: bool = False):
    """
    Load Gemma-3-270M-IT in 4-bit QLoRA mode.
    If lora=True also wraps with LoRA adapters for training.
    """
    log.info(f"Loading {MODEL_ID} (lora={lora}) …")
    tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"

    mdl = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=BNB_CFG,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.float32,
    )
    mdl.config.use_cache = False

    if lora:
        mdl = prepare_model_for_kbit_training(mdl, use_gradient_checkpointing=True)
        mdl = get_peft_model(mdl, LORA_CFG)
        mdl.print_trainable_parameters()

    log.info(f"GPU after load : "
             f"{torch.cuda.memory_allocated()/1e9:.2f} GB")
    return mdl, tok


def free(model=None):
    if model is not None:
        del model
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    log.info(f"GPU after free : {torch.cuda.memory_allocated()/1e9:.2f} GB")


print("Model loader and logger defined.")

Model loader and logger defined.


## Cell 9 — Evaluation Runner

In [15]:
@torch.no_grad()
def run_evaluation(model, tokenizer, samples: List[Dict],
                   phase: str, n: int = N_EVAL_SAMPLES) -> Dict:
    """
    Generate answers for `n` random test samples, compute metrics,
    log to CSV, and return the metric dict.
    """
    metrics = {"rouge1": 0, "rouge2": 0, "rougeL": 0, "f1": 0, "n": 0}
    
    model.eval()
    subset = random.sample(samples, min(n, len(samples)))
    preds, refs = [], []

    log.info(f"--- Evaluation [{phase}] on {len(subset)} samples ---")

    for i, s in enumerate(subset):
        prompt = extract_prompt(s["text"])
        ref    = s["answer"]

        enc = tokenizer(
            prompt, return_tensors="pt", truncation=True,
            max_length=MAX_SEQ_LEN - MAX_NEW_TOKENS,
        ).to(model.device)

        outputs = model(**enc)
        logits = outputs.logits
        if torch.isnan(logits).any() or torch.isinf(logits).any():
            log.error(f"Sample {i}: Detected NaN/Inf in logits! This will crash the GPU.")
            # Optional: print the prompt to see if a specific character caused it
            # print(f"Problematic prompt: {prompt}") 
            continue # Skip this sample to prevent the crash

        gen = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=0.1,
            pad_token_id=tokenizer.eos_token_id,
        )
        new_ids  = gen[0][enc["input_ids"].shape[1]:]
        raw_text = tokenizer.decode(new_ids, skip_special_tokens=True)
        pred     = extract_generated(raw_text)

        preds.append(pred)
        refs.append(ref)

        if (i + 1) % 50 == 0:
            log.info(f"  {i+1}/{len(subset)} done")

    metrics = compute_metrics(preds, refs)
    metrics["phase"] = phase

    _append_csv(EVAL_LOG_PATH, EVAL_FIELDS, metrics)

    log.info(
        f"[{phase}] BLEU:{metrics['bleu']:.4f}  "
        f"R1:{metrics['rouge1']:.4f}  R2:{metrics['rouge2']:.4f}  "
        f"RL:{metrics['rougeL']:.4f}  F1:{metrics['f1']:.4f}  "
        f"n={metrics['n']}"
    )
    return metrics


print("Evaluation runner defined.")

Evaluation runner defined.


## Cell 10 — PRE-Finetuning Evaluation (Baseline)

In [ ]:
log.info("=" * 55)
log.info("STEP 1 — PRE-FINETUNING EVALUATION")
log.info("=" * 55)

base_model, base_tok = load_model_and_tokenizer(lora=False)
pre_metrics = run_evaluation(base_model, base_tok, test_samples, phase="pre")

# Print readable table
print("\n" + "="*55)
print(" PRE-FINETUNING BASELINE")
print("="*55)
for k, v in pre_metrics.items():
    if k != "phase":
        print(f"  {k:<10}: {v}")
print("="*55)

with open(RESULTS_DIR / "pre_metrics.json", "w") as f:
    json.dump(pre_metrics, f, indent=2)

# Unload — we reload fresh for training
free(base_model)
del base_model, base_tok
print("Base model unloaded.")

## Cell 11 — QLoRA Finetuning

In [17]:
import gc
import torch

def clean_vram():
    if 'ft_model' in globals():
        # Only delete if we are about to re-load it
        # del ft_model 
        pass
    gc.collect()
    torch.cuda.empty_cache()
    # If using multiple GPUs, clear both
    torch.cuda.set_device(0)
    torch.cuda.empty_cache()
    
clean_vram()

In [18]:
import torch

# Monkey-patch the device count so Trainer only sees one GPU
torch.cuda.device_count = lambda: 1

# If you already loaded the model, consolidate it onto one device
if 'ft_model' in globals():
    print("Moving model to cuda:0...")
    ft_model.to("cuda:0")

print(f"Fake Device Count: {torch.cuda.device_count()}")

Fake Device Count: 1


In [ ]:
log.info("=" * 55)
log.info("STEP 2 — QLORA FINETUNING")
log.info(f"Training samples : {len(train_hf)}")
log.info("=" * 55)

ADAPTER_DIR = str(CKPT_DIR / "lora_adapter")

ft_model, ft_tok = load_model_and_tokenizer(lora=True)

# sft_config = SFTConfig(
#     output_dir=str(CKPT_DIR / "sft_output"),
#     num_train_epochs=TRAIN_CFG["num_train_epochs"],
#     per_device_train_batch_size=TRAIN_CFG["per_device_train_batch_size"],
#     gradient_accumulation_steps=TRAIN_CFG["gradient_accumulation_steps"],
#     gradient_checkpointing=True,
#     learning_rate=TRAIN_CFG["learning_rate"],
#     warmup_ratio=TRAIN_CFG["warmup_ratio"],
#     lr_scheduler_type=TRAIN_CFG["lr_scheduler_type"],
#     logging_steps=TRAIN_CFG["logging_steps"],
#     save_strategy=TRAIN_CFG["save_strategy"],
#     save_total_limit=TRAIN_CFG["save_total_limit"],
#     fp16=TRAIN_CFG["fp16"],
#     optim=TRAIN_CFG["optim"],
#     max_grad_norm=TRAIN_CFG["max_grad_norm"],
#     weight_decay=TRAIN_CFG["weight_decay"],
#     report_to=TRAIN_CFG["report_to"],
#     dataloader_num_workers=TRAIN_CFG["dataloader_num_workers"],
#     # SFT-specific
#     max_length=MAX_SEQ_LEN,
#     dataset_text_field="text",
#     packing=False,
#     seed=SEED,
# )

# trainer = SFTTrainer(
#     model=ft_model,
#     processing_class=ft_tok,      # trl >= 0.9 API
#     train_dataset=train_hf,
#     args=sft_config,
#     callbacks=[StepLogger()],
# )

# t0 = time.time()
# result = trainer.train()
# elapsed = time.time() - t0

# log.info(f"Training finished in {elapsed/60:.1f} min")
# log.info(f"Final loss : {result.training_loss:.4f}")

# # Save LoRA adapter
# ft_model.save_pretrained(ADAPTER_DIR)
# ft_tok.save_pretrained(ADAPTER_DIR)
# log.info(f"LoRA adapter saved → {ADAPTER_DIR}")

In [ ]:
# # 1. Force the existing model to float16 (Crucial for T4)
# ft_model.to(torch.float16) 

# # 2. Update Config
# sft_config = SFTConfig(
#     output_dir=str(CKPT_DIR / "sft_output"),
#     num_train_epochs=TRAIN_CFG["num_train_epochs"],
#     per_device_train_batch_size=TRAIN_CFG["per_device_train_batch_size"],
#     gradient_accumulation_steps=TRAIN_CFG["gradient_accumulation_steps"],
#     gradient_checkpointing=True,
#     learning_rate=TRAIN_CFG["learning_rate"],
#     warmup_ratio=TRAIN_CFG["warmup_ratio"],
#     lr_scheduler_type=TRAIN_CFG["lr_scheduler_type"],
#     logging_steps=TRAIN_CFG["logging_steps"],
#     save_strategy=TRAIN_CFG["save_strategy"],
#     save_total_limit=TRAIN_CFG["save_total_limit"],
    
#     # --- MANDATORY T4 CHANGES ---
#     fp16=True,                # Enable standard half-precision
#     bf16=False,               # Explicitly disable bfloat16
#     optim="paged_adamw_32bit",# Best optimizer for QLoRA on T4
#     # ----------------------------
    
#     max_grad_norm=TRAIN_CFG["max_grad_norm"],
#     weight_decay=TRAIN_CFG["weight_decay"],
#     report_to=TRAIN_CFG["report_to"],
#     dataloader_num_workers=TRAIN_CFG["dataloader_num_workers"],
#     max_length=MAX_SEQ_LEN,
#     dataset_text_field="text",
#     packing=False,
#     seed=SEED,
# )

# # 3. Re-initialize Trainer
# trainer = SFTTrainer(
#     model=ft_model,
#     processing_class=ft_tok,
#     train_dataset=train_hf,
#     args=sft_config,
#     callbacks=[StepLogger()],
# )

# # 4. Train
# t0 = time.time()
# result = trainer.train()

In [ ]:
import torch
import gc

# 1. Force-cast all trainable parameters and buffers to float32
# This targets the LoRA adapters specifically, which are likely the BF16 culprits
for name, param in ft_model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)
    if param.grad is not None:
        param.grad.data = param.grad.data.to(torch.float32)

for buffer in ft_model.buffers():
    buffer.data = buffer.data.to(torch.float32)

# 2. Hard-reset the config to disable ALL scaling logic
sft_config = SFTConfig(
    output_dir=str(CKPT_DIR / "sft_output"),
    num_train_epochs=TRAIN_CFG["num_train_epochs"],
    per_device_train_batch_size=TRAIN_CFG["per_device_train_batch_size"],
    gradient_accumulation_steps=TRAIN_CFG["gradient_accumulation_steps"],
    gradient_checkpointing=True,
    learning_rate=TRAIN_CFG["learning_rate"],
    logging_steps=1,
    
    # CRITICAL: These must be False to stop the NotImplementedError
    fp16=False, 
    bf16=False, 
    
    # Use a standard optimizer that doesn't rely on half-precision kernels
    optim="adamw_torch", 
    
    max_grad_norm=TRAIN_CFG["max_grad_norm"],
    max_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    packing=False,
    seed=SEED,
)

# 3. Wipe the old trainer object to clear the internal GradScaler
if 'trainer' in globals():
    del trainer
gc.collect()
torch.cuda.empty_cache()

# 4. Re-initialize a fresh Trainer
trainer = SFTTrainer(
    model=ft_model,
    processing_class=ft_tok,
    train_dataset=train_hf,
    args=sft_config,
    callbacks=[StepLogger()],
)

# 5. Start training
log.info("Starting training: Force Float32 / No Scaler")
trainer.train()

## Cell 12 — POST-Finetuning Evaluation

In [ ]:
log.info("=" * 55)
log.info("STEP 3 — POST-FINETUNING EVALUATION")
log.info("=" * 55)

post_metrics = run_evaluation(ft_model, ft_tok, test_samples, phase="post")

with open(RESULTS_DIR / "post_metrics.json", "w") as f:
    json.dump(post_metrics, f, indent=2)

# Free after eval is done
free(ft_model)
del ft_model, trainer, ft_tok
print("Finetuned model unloaded.")

## Cell 13 — Results Comparison Table

In [ ]:
# METRICS = ["bleu", "rouge1", "rouge2", "rougeL", "f1"]

# rows = []
# for m in METRICS:
#     pre_v  = pre_metrics.get(m, 0.0)
#     post_v = post_metrics.get(m, 0.0)
#     delta  = post_v - pre_v
#     pct    = (delta / pre_v * 100) if pre_v > 0 else float("nan")
#     rows.append({"metric": m, "pre": pre_v, "post": post_v,
#                  "delta": delta, "delta_pct": pct})

# df = pd.DataFrame(rows).set_index("metric")
# df.to_csv(RESULTS_DIR / "comparison.csv")

# # ── Pretty print ─────────────────────────────────────────────────
# W = 62
# print("\n" + "="*W)
# print(f"  {'METRIC':<10} {'PRE':>9} {'POST':>9} {'Δ':>9} {'Δ%':>9}")
# print("-"*W)
# for _, row in df.iterrows():
#     pct_s = f"{row.delta_pct:+.1f}%" if not pd.isna(row.delta_pct) else "N/A"
#     print(f"  {row.name:<10} {row.pre:>9.4f} {row.post:>9.4f} "
#           f"{row.delta:>+9.4f} {pct_s:>9}")
# print("="*W)

# print(f"\nOutputs saved to {RESULTS_DIR}")
# print(f"  comparison.csv   — metric comparison table")
# print(f"  pre_metrics.json — raw pre-eval scores")
# print(f"  post_metrics.json— raw post-eval scores")
# print(f"  {LOG_DIR.name}/training_log.csv — step-level loss")
# print(f"  {LOG_DIR.name}/eval_log.csv     — eval log")

## Cell 14 — Quick Inference Test (Optional)

In [ ]:
# Reload the saved adapter for a quick sanity-check
inf_model, inf_tok = load_model_and_tokenizer(lora=False)
inf_model = PeftModel.from_pretrained(inf_model, ADAPTER_DIR)
inf_model.eval()


def ask(question: str, context: str = "General medical knowledge.") -> str:
    prompt = (
        f"<instruction>Answer the following medical question clearly and accurately.</instruction>\n"
        f"<context>{context}</context>\n"
        f"<input>{question}</input>\n"
        f"<output>"
    )
    enc = inf_tok(prompt, return_tensors="pt", truncation=True,
                  max_length=MAX_SEQ_LEN - MAX_NEW_TOKENS).to(inf_model.device)
    with torch.no_grad():
        out = inf_model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS,
                                  do_sample=True, temperature=0.1,
                                  pad_token_id=inf_tok.eos_token_id)
    new = out[0][enc["input_ids"].shape[1]:]
    return extract_generated(inf_tok.decode(new, skip_special_tokens=True))


q = "What is the first-line treatment for type 2 diabetes?"
print(f"Q: {q}")
print(f"A: {ask(q)}")